# MachSense - Day 8: Explainable AI (SHAP)
## Global Feature Importance, Local Instance Attribution, and Non-Causal Operator Diagnostics

**Objective**: Apply SHAP (SHapley Additive exPlanations) to the registered champion model (`v1.0.0`) to explain global feature importance, decompose individual predictions into risk escalators and stabilizers, translate mathematical attribution into plain-English operator alerts, and document non-causal safety boundaries.

### 1. Environment Setup & Champion Artifact Loading

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from machsense.config.settings import get_settings
from machsense.models.explainability import MachSenseExplainer
from machsense.models.registry import ModelRegistry

settings = get_settings()
registry = ModelRegistry(registry_dir=PROJECT_ROOT / "models")
champion_model, preprocessor, metadata = registry.load_champion_model()
print(f"Loaded Champion Model: {metadata.model_name} (Version: {metadata.model_version})")
print(f"Model Type: {metadata.model_type}")
print(f"Optimal Operational Threshold: {metadata.optimal_threshold:.4f}")

### 2. Loading Test Set & Initializing TreeExplainer

In [ ]:
# Load processed evaluation test set
X_test = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "X_test_transformed.csv")
y_test = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "y_test.csv").squeeze()
failure_modes_test = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "failure_modes_test.csv")

print(f"X_test shape: {X_test.shape}")
print(f"Total Test Failures: {int(y_test.sum())} / {len(y_test)}")

# Initialize Explainer
explainer = MachSenseExplainer(model=champion_model, feature_names=list(X_test.columns))
print(f"Explainer Expected (Baseline) Failure Probability: {explainer.expected_value:.4f}")

### 3. Global Feature Importance (Mean Absolute SHAP)

In [ ]:
# Calculate Global Importance Table
global_imp = explainer.compute_global_importance(X_test)
display(global_imp)

# Generate Global Importance Bar Plot
fig_bar = explainer.plot_global_importance(X_test, max_display=10)
plt.show()

### 4. Global SHAP Summary / Beeswarm Distribution
The beeswarm plot illustrates not only the magnitude of importance, but also the direction of effect (e.g. high values of `torque_nm` and `power_w` shifting risk positively to the right).

In [ ]:
shap_exp = explainer.get_shap_explanation(X_test)
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_exp, max_display=10, show=False)
plt.title("MachSense SHAP Beeswarm Summary Plot", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

### 5. Local Instance Attribution & Operator Diagnostics
We analyze specific failure case studies representing distinct failure physics:

#### Case Study A: Heat Dissipation Failure (HDF)

In [ ]:
hdf_indices = np.where(failure_modes_test["heat_dissipation_failure"] == 1)[0]
if len(hdf_indices) > 0:
    hdf_idx = int(hdf_indices[0])
    print(f"Evaluating HDF Instance (Test Sample #{hdf_idx}):")
    hdf_diag = explainer.explain_instance(X_test.iloc[hdf_idx], threshold=metadata.optimal_threshold)
    print(hdf_diag.operator_summary)
    explainer.plot_waterfall(X_test.iloc[hdf_idx], title="Local Attribution: Heat Dissipation Failure (HDF)")
    plt.show()

#### Case Study B: Power Failure (PWF)

In [ ]:
pwf_indices = np.where(failure_modes_test["power_failure"] == 1)[0]
if len(pwf_indices) > 0:
    pwf_idx = int(pwf_indices[0])
    print(f"Evaluating PWF Instance (Test Sample #{pwf_idx}):")
    pwf_diag = explainer.explain_instance(X_test.iloc[pwf_idx], threshold=metadata.optimal_threshold)
    print(pwf_diag.operator_summary)
    explainer.plot_waterfall(X_test.iloc[pwf_idx], title="Local Attribution: Power Failure (PWF)")
    plt.show()

#### Case Study C: Overstrain Failure (OSF)

In [ ]:
osf_indices = np.where(failure_modes_test["overstrain_failure"] == 1)[0]
if len(osf_indices) > 0:
    osf_idx = int(osf_indices[0])
    print(f"Evaluating OSF Instance (Test Sample #{osf_idx}):")
    osf_diag = explainer.explain_instance(X_test.iloc[osf_idx], threshold=metadata.optimal_threshold)
    print(osf_diag.operator_summary)
    explainer.plot_waterfall(X_test.iloc[osf_idx], title="Local Attribution: Overstrain Failure (OSF)")
    plt.show()

#### Case Study D: Nominal Healthy Machine Cycle

In [ ]:
nom_indices = np.where(y_test == 0)[0]
nom_idx = int(nom_indices[0])
print(f"Evaluating Nominal Instance (Test Sample #{nom_idx}):")
nom_diag = explainer.explain_instance(X_test.iloc[nom_idx], threshold=metadata.optimal_threshold)
print(nom_diag.operator_summary)
explainer.plot_waterfall(X_test.iloc[nom_idx], title="Local Attribution: Nominal Healthy State")
plt.show()

### 6. Conclusions & Non-Causal Safety Protocols
1. **Physics Feature Dominance**: Engineered features (`power_w`, `overstrain_index`, `temp_difference_k`) are among the highest contributors across all critical failure categories.
2. **Actionable Root Slicing**: Operators can immediately distinguish between electrical overload, cooling degradation, and tool mechanical wear.
3. **Non-Causal Precaution**: All automated alerts include clear non-causal disclaimers instructing operators to visually inspect physical systems before executing high-impact maintenance operations.